# Pig Posture Recognition – V3 Inference

**Verbesserungen:**
1. **Test-Time BN Adaptation** – BatchNorm-Statistiken auf Testdaten neu berechnen
2. **TTA mit Flip-Korrektur** – Links/Rechts-Wahrscheinlichkeiten korrekt tauschen
3. **Multi-Fold Ensemble** – Alle Fold-Modelle gemittelt
4. **Pseudo-Label Export** – High-Confidence Predictions fuer Self-Training

## Configuration

In [1]:
import os

TAG = "T2"

_candidates = [
    'multiview_pig_posture_recognition',
    './multiview_pig_posture_recognition',
    '/datasets/multi-view-pig-posture-recognition',
    '/multi-view-pig-posture-recognition',
]
DATA_ROOT = None
for _p in _candidates:
    if os.path.isdir(_p):
        DATA_ROOT = _p
        break
assert DATA_ROOT is not None, f'Datenverzeichnis nicht gefunden!'
print(f'DATA_ROOT = {os.path.abspath(DATA_ROOT)}')

TEST_CSV  = os.path.join(DATA_ROOT, 'test.csv')
IMG_DIR   = os.path.join(DATA_ROOT, 'test_images')

CKPT_DIR  = f"runs/v3_{TAG.lower()}"
CKPT_PATHS = sorted([
    os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR)
    if f.startswith("best_model_fold_")
])

OUTPUT_FILE = f"{TAG}_v3_submission.csv"

IMG_SIZE     = 392              # 28*14, wird aus Checkpoint uebernommen falls anders
BATCH_SIZE   = 32
NUM_WORKERS  = 8
USE_TTA      = True
PAD_RATIO    = 0.1              # Angepasst an Training
NUM_CLASSES  = 5

CLASS_NAMES = ['Lateral_lying_left', 'Lateral_lying_right',
               'Sitting', 'Standing', 'Sternal_lying']

# ─── Test-Time BN Adaptation ───
ADAPT_BN         = True
BN_ADAPT_BATCHES = 50     # Anzahl Batches fuer BN-Statistik-Update

# ─── Pseudo-Labeling ───
EXPORT_PSEUDO_LABELS = True
PSEUDO_THRESHOLD     = 0.95
PSEUDO_OUTPUT        = f"pseudo_labels_{TAG.lower()}.csv"

print(f"Tag: {TAG}  |  TTA: {USE_TTA}  |  BN-Adapt: {ADAPT_BN}")
print(f"Modelle: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    print(f"  - {os.path.basename(p)}")

DATA_ROOT = /datasets/multi-view-pig-posture-recognition
Tag: T2  |  TTA: True  |  BN-Adapt: True
Modelle: 3
  - best_model_fold_1.pth
  - best_model_fold_2.pth
  - best_model_fold_3.pth


## Imports

In [2]:
import os, ast
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

<jemalloc>: Unsupported system page size


Device: cuda


In [3]:
# ─── Settings aus erstem Checkpoint uebernehmen ───
if CKPT_PATHS:
    _ckpt = torch.load(CKPT_PATHS[0], map_location='cpu')
    _ckpt_img_size = _ckpt.get('img_size', IMG_SIZE)
    _ckpt_pad_ratio = _ckpt.get('pad_ratio', PAD_RATIO)

    if _ckpt_img_size != IMG_SIZE:
        print(f"IMG_SIZE aus Checkpoint uebernommen: {_ckpt_img_size} (Config war: {IMG_SIZE})")
        IMG_SIZE = _ckpt_img_size
    if _ckpt_pad_ratio != PAD_RATIO:
        print(f"PAD_RATIO aus Checkpoint uebernommen: {_ckpt_pad_ratio} (Config war: {PAD_RATIO})")
        PAD_RATIO = _ckpt_pad_ratio

    print(f"Checkpoint-Info: model={_ckpt.get('model_name','?')}, "
          f"val_f1={_ckpt.get('val_f1',0):.4f}, "
          f"cam={_ckpt.get('val_camera','?')}")
    del _ckpt
else:
    print("Keine Checkpoints gefunden!")

Checkpoint-Info: model=vit_base_patch14_dinov2.lvd142m, val_f1=0.8931, cam=pen1_tur_cam1


## Test Dataset

In [4]:
class PigTestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.25):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px));  y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px));  y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(os.path.join(self.img_dir, row['image_id'])).convert('RGB')
        crop = self._crop(img, row['bbox'])
        if self.transform:
            crop = self.transform(crop)
        return crop, row['row_id']

## Test-Time BN Adaptation

Fuer Modelle mit BatchNorm (ConvNeXt, EfficientNet, etc.):
Die BN-Statistiken (mean/variance) wurden auf Trainingsdaten berechnet.
Die Testdaten kommen von ANDEREN Kameras → andere Pixelverteilung.

**Loesung:** BN-Statistiken auf den Testdaten neu berechnen, bevor wir predicten.

Fuer DINOv2 (nutzt LayerNorm statt BatchNorm) wird dieser Schritt automatisch uebersprungen.

In [5]:
def adapt_batch_norm(model, loader, device, n_batches=50):
    """Recalculate BatchNorm statistics on test data."""
    bn_layers = [m for m in model.modules()
                 if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.SyncBatchNorm))]

    if not bn_layers:
        print("    Keine BatchNorm Layer (z.B. DINOv2/ViT nutzt LayerNorm) – uebersprungen.")
        return

    # Reset running statistics
    for bn in bn_layers:
        bn.running_mean.zero_()
        bn.running_var.fill_(1)
        bn.momentum = None   # Cumulative moving average statt exponential

    model.train()  # BN in train mode → berechnet neue Stats
    with torch.no_grad():
        for i, (imgs, _) in enumerate(tqdm(loader, desc="    BN-Adapt", leave=False)):
            if i >= n_batches:
                break
            model(imgs.to(device))
    model.eval()

    print(f"    BN auf {min(n_batches, len(loader))} Batches adaptiert ({len(bn_layers)} BN Layer)")

## TTA mit Flip-Korrektur

Bei horizontal gespiegelten Views werden die Wahrscheinlichkeiten
fuer Klasse 0 (left) und 1 (right) getauscht: `p[:, [0,1]] = p[:, [1,0]]`

In [6]:
S = IMG_SIZE
NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

TTA_CONFIGS = [
    (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                T.ToTensor(), T.Normalize(*NORM)]), False),
    (T.Compose([T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                T.RandomHorizontalFlip(p=1.0),
                T.ToTensor(), T.Normalize(*NORM)]), True),
    (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                T.CenterCrop(S),
                T.ToTensor(), T.Normalize(*NORM)]), False),
    (T.Compose([T.Resize((S+32, S+32), interpolation=T.InterpolationMode.BICUBIC),
                T.CenterCrop(S), T.RandomHorizontalFlip(p=1.0),
                T.ToTensor(), T.Normalize(*NORM)]), True),
    (T.Compose([T.Resize((S+64, S+64), interpolation=T.InterpolationMode.BICUBIC),
                T.CenterCrop(S),
                T.ToTensor(), T.Normalize(*NORM)]), False),
    (T.Compose([T.Resize((int(S*0.75), int(S*0.75))),
                T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
                T.ToTensor(), T.Normalize(*NORM)]), False),
]

print(f"TTA: {len(TTA_CONFIGS)} Views ({sum(1 for _,f in TTA_CONFIGS if f)} mit Flip-Korrektur)")

TTA: 6 Views (2 mit Flip-Korrektur)


## Inference

In [7]:
test_df = pd.read_csv(TEST_CSV)
print(f"Test-Instanzen: {len(test_df)}")

@torch.no_grad()
def predict_tta(model, df, img_dir, tta_configs):
    all_probs = []
    for i, (tf, is_flipped) in enumerate(tta_configs):
        ds = PigTestDataset(df, img_dir, transform=tf, pad_ratio=PAD_RATIO)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        probs = []
        for imgs, _ in tqdm(loader, desc=f'  TTA {i+1}/{len(tta_configs)}', leave=False):
            with autocast():
                logits = model(imgs.to(DEVICE))
            p = F.softmax(logits, dim=1).cpu().numpy()
            if is_flipped:
                p[:, [0, 1]] = p[:, [1, 0]]
            probs.append(p)
        all_probs.append(np.vstack(probs))
    return np.mean(all_probs, axis=0)


# ─── Loader fuer BN Adaptation (ohne Augmentierung) ───
S = IMG_SIZE
NORM = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]

bn_transform = T.Compose([
    T.Resize((S, S), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(), T.Normalize(*NORM),
])
bn_ds = PigTestDataset(test_df, IMG_DIR, transform=bn_transform, pad_ratio=PAD_RATIO)
bn_loader = DataLoader(bn_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True)

configs = TTA_CONFIGS if USE_TTA else [TTA_CONFIGS[0]]
ensemble_probs = []

for fold, path in enumerate(CKPT_PATHS):
    print(f"\nModell {fold+1}/{len(CKPT_PATHS)}: {os.path.basename(path)}")
    ckpt = torch.load(path, map_location='cpu')
    name = ckpt.get('model_name', 'convnext_base')
    ckpt_img_size = ckpt.get('img_size', IMG_SIZE)
    print(f"  Architektur: {name}  |  Val F1: {ckpt.get('val_f1', 0):.4f}  |  img_size: {ckpt_img_size}")

    model = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES, img_size=ckpt_img_size)
    model.load_state_dict(ckpt['model'])
    model.to(DEVICE).eval()

    # ─── Test-Time BN Adaptation ───
    if ADAPT_BN:
        adapt_batch_norm(model, bn_loader, DEVICE, n_batches=BN_ADAPT_BATCHES)

    fold_probs = predict_tta(model, test_df, IMG_DIR, configs)
    ensemble_probs.append(fold_probs)

    del model
    torch.cuda.empty_cache()

final_probs = np.mean(ensemble_probs, axis=0)
predictions = final_probs.argmax(axis=1)

print(f"\n{len(predictions)} Vorhersagen aus {len(CKPT_PATHS)} Modellen")

Test-Instanzen: 11708

Modell 1/3: best_model_fold_1.pth
  Architektur: vit_base_patch14_dinov2.lvd142m  |  Val F1: 0.8931  |  img_size: 392
    Keine BatchNorm Layer (z.B. DINOv2/ViT nutzt LayerNorm) – uebersprungen.


  TTA 1/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 2/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

...

  TTA 3/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 4/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 5/6:   0%|          | 0/366 [00:00<?, ?it/s]

  TTA 6/6:   0%|          | 0/366 [00:00<?, ?it/s]


11708 Vorhersagen aus 3 Modellen


## Submission

In [8]:
submission = pd.DataFrame({
    'row_id': test_df['row_id'].values,
    'class_id': predictions.astype(int)
})
submission.to_csv(OUTPUT_FILE, index=False)

assert list(submission.columns) == ['row_id', 'class_id']
assert set(submission['class_id'].unique()).issubset(set(range(5)))
assert len(submission) == len(test_df)

print(f"✓ Submission: {OUTPUT_FILE} ({len(submission)} Zeilen)")
print(f"\nVerteilung:")
for c in range(NUM_CLASSES):
    cnt = (submission['class_id'] == c).sum()
    pct = 100 * cnt / len(submission)
    bar = '█' * int(30 * cnt / len(submission))
    print(f"  {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5} ({pct:.1f}%)")

submission.head(10)

✓ Submission: T2_v3_submission.csv (11708 Zeilen)

Verteilung:
  0 - Lateral_lying_left     ███                             1206 (10.3%)
  1 - Lateral_lying_right    ███                             1402 (12.0%)
  2 - Sitting                █                                434 (3.7%)
  3 - Standing               ███████████████                 6040 (51.6%)
  4 - Sternal_lying          ██████                          2626 (22.4%)


,row_id,class_id
0,test_pen1_tur_cam1_20250920_174649_0000,3
1,test_pen1_tur_cam1_20250920_174649_0001,1
2,test_pen1_tur_cam1_20250920_174649_0002,1
3,test_pen1_tur_cam1_20250920_174649_0003,1
4,test_pen1_tur_cam1_20250920_174649_0004,0
5,test_pen1_tur_cam1_20250920_174649_0005,1
6,test_pen1_tur_cam1_20250920_174649_0006,1
7,test_pen1_tur_cam1_20250920_174649_0007,1
8,test_pen1_tur_cam1_20250920_174649_0008,0
9,test_pen1_tur_cam1_20250921_050022_0000,0


## Pseudo-Label Export

Exportiert high-confidence Test-Predictions als CSV fuer iteratives Self-Training.

**Workflow:**
1. Hier exportieren (→ `pseudo_labels_t1.csv`)
2. In `v3_train.ipynb`: `USE_PSEUDO_LABELS = True`, `PSEUDO_CSV = "pseudo_labels_t1.csv"`
3. Neu trainieren → Modell lernt Test-Domain
4. Wiederholen (2-3 Iterationen)

In [9]:
if EXPORT_PSEUDO_LABELS:
    max_probs = final_probs.max(axis=1)
    mask = max_probs >= PSEUDO_THRESHOLD

    pseudo_df = test_df[mask].copy()
    pseudo_df['class_id'] = predictions[mask].astype(int)
    pseudo_df['confidence'] = max_probs[mask]
    pseudo_df.to_csv(PSEUDO_OUTPUT, index=False)

    n = mask.sum()
    print(f"Pseudo-Labels: {PSEUDO_OUTPUT}")
    print(f"  {n} von {len(test_df)} ({100*n/len(test_df):.1f}%) ueber Threshold {PSEUDO_THRESHOLD}")
    print(f"\n  Verteilung:")
    for c in range(NUM_CLASSES):
        cnt = (pseudo_df['class_id'] == c).sum()
        print(f"    {c} - {CLASS_NAMES[c]:<22} {cnt:>5}")
    print(f"\n  Naechster Schritt: v3_train.ipynb mit USE_PSEUDO_LABELS=True starten")
else:
    print("Pseudo-Label Export deaktiviert.")

Pseudo-Labels: pseudo_labels_t2.csv
  110 von 11708 (0.9%) ueber Threshold 0.95

  Verteilung:
    0 - Lateral_lying_left         0
    1 - Lateral_lying_right        0
    2 - Sitting                  110
    3 - Standing                   0
    4 - Sternal_lying              0

  Naechster Schritt: v3_train.ipynb mit USE_PSEUDO_LABELS=True starten
